In [ ]:
from pathlib import Path
import pandas as pd

df_llm = pd.read_parquet(Path("../data/processed/llm_structured_coding.parquet"))
df_base = pd.read_parquet(Path("../data/processed/baseline_features.parquet"))

cols = [
    "assessment_id",
    "needs",
    "urgent_needs",
    "notes",
    "gt_need_categories",
    "pred_need_categories",
    "llm_need_categories",
    "gt_urgent_categories",
    "pred_urgent_need_categories",
    "llm_urgent_need_categories",
]

df_compare = df_llm.merge(
    df_base[["assessment_id", "pred_need_categories", "pred_urgent_need_categories"]],
    on="assessment_id",
    how="left"
)

df_compare[cols].head(10)

In [ ]:
df_llm["llm_need_categories"].value_counts().head(20)
df_llm["llm_urgent_need_categories"].value_counts().head(20)
df_llm["llm_parse_error"].value_counts(dropna=False)
df_llm[[
    "assessment_id",
    "gt_need_categories",
    "llm_need_categories",
    "gt_urgent_categories",
    "llm_urgent_need_categories"
]].head(50)

In [ ]:
def normalize_pipe_set(s):
    if pd.isna(s) or s == "":
        return set()
    return set(s.split("|"))

df_llm["need_exact_match"] = df_llm.apply(
    lambda r: normalize_pipe_set(r["llm_need_categories"]) == normalize_pipe_set(r["gt_need_categories"]),
    axis=1
)

df_llm["urgent_exact_match"] = df_llm.apply(
    lambda r: normalize_pipe_set(r["llm_urgent_need_categories"]) == normalize_pipe_set(r["gt_urgent_categories"]),
    axis=1
)

df_llm["need_exact_match"].mean(), df_llm["urgent_exact_match"].mean()

In [ ]:
df_compare["baseline_need_exact_match"] = df_compare.apply(
    lambda r: normalize_pipe_set(r["pred_need_categories"]) == normalize_pipe_set(r["gt_need_categories"]),
    axis=1
)

df_compare["baseline_urgent_exact_match"] = df_compare.apply(
    lambda r: normalize_pipe_set(r["pred_urgent_need_categories"]) == normalize_pipe_set(r["gt_urgent_categories"]),
    axis=1
)

## Step 2: notebook quick check sui flag

In [ ]:
df_llm = pd.read_parquet(Path("../data/processed/llm_structured_coding.parquet"))
df_base = pd.read_parquet(Path("../data/processed/baseline_features.parquet"))

df_compare = df_llm.merge(
    df_base[
        [
            "assessment_id",
            "pred_displacement",
            "pred_children_present",
            "pred_elderly_present",
            "pred_disability_present",
            "pred_health_issue",
            "pred_access_constraint",
        ]
    ],
    on="assessment_id",
    how="left"
)

flag_cols = [
    ("gt_displacement", "pred_displacement", "llm_displacement"),
    ("gt_children_present", "pred_children_present", "llm_children_present"),
    ("gt_elderly_present", "pred_elderly_present", "llm_elderly_present"),
    ("gt_disability_present", "pred_disability_present", "llm_disability_present"),
    ("gt_health_issue", "pred_health_issue", "llm_health_issue"),
    ("gt_access_constraint", "pred_access_constraint", "llm_access_constraint"),
]

for gt_col, base_col, llm_col in flag_cols:
    print(f"\n=== {gt_col} ===")
    print("Baseline accuracy:", (df_compare[gt_col] == df_compare[base_col]).mean())
    print("LLM accuracy:", (df_compare[gt_col] == df_compare[llm_col]).mean())